# 01 · Datos y exploración de retornos

Paso 1: descarga de precios diarios **ajustados** (dividendos y splits) de SPY, QQQ, TLT, GLD y EEM, 2005-01-01 a 2025-12-31, con caché local en `data/raw/prices.parquet` (no se sube a Git).

Fuente: Yahoo Finance vía `yfinance` (uso personal/educativo).

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.data import download_prices, to_returns

px = download_prices(cache="../data/raw/prices.parquet")
print(px.shape, px.index.min().date(), px.index.max().date())
print(px.isna().sum())                                  # nulos por ETF
print(px.index.is_monotonic_increasing, px.index.duplicated().sum())
rets = to_returns(px)
print(rets.abs().max())                                 # ningún retorno diario absurdo
print(rets.abs().stack().nlargest(5))                   # los 5 días más extremos

(5282, 5) 2005-01-03 2025-12-30
Ticker
SPY    0
QQQ    0
TLT    0
GLD    0
EEM    0
dtype: int64
True 0
Ticker
SPY    0.145198
QQQ    0.121648
TLT    0.075195
GLD    0.112905
EEM    0.227698
dtype: float64
Date        Ticker
2008-10-13  EEM       0.227698
2008-10-28  EEM       0.209327
2008-10-15  EEM       0.161662
2008-10-13  SPY       0.145198
2008-11-21  EEM       0.144031
dtype: float64


## Notas de calidad de datos (Paso 1)

- **Tamaño:** 5,282 días de negociación × 5 ETFs, del 2005-01-03 al 2025-12-30. **0 nulos**, índice ordenado y sin fechas duplicadas.
- **Fecha final:** `end` en `yfinance` es exclusivo, así que `end="2025-12-31"` termina el 30 de diciembre de 2025. Es una fecha fija: cualquiera obtiene la misma muestra.
- **Días extremos:** los 5 movimientos absolutos más grandes son de la crisis de octubre-noviembre de 2008 (EEM +22.8 % el 2008-10-13; SPY +14.5 % el mismo día). Las peores caídas de SPY, QQQ y TLT son de marzo de 2020 y la de GLD es la del 2013-04-15. Todos coinciden con eventos conocidos: **no hay errores de datos evidentes**.
- **Sesgo de selección del universo:** los 5 ETFs se eligieron hoy, sabiendo que sobrevivieron y son líquidos. Es una limitación real del estudio.
- **Datos:** Yahoo Finance vía `yfinance`, precios ajustados (`auto_adjust=True`), descargados el 2026-11-25. Huella de la caché: `sha256[:12] = 49ae3c57684c`. Yahoo puede recalcular precios ajustados, así que una descarga futura podría diferir un poco. Los datos **no** se suben al repositorio.


## Paso 2 · Exploración de retornos y hechos estilizados

### 2.1 Buy & hold y drawdown
Es el benchmark que toda estrategia debe superar.

In [2]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rets = rets.dropna()
growth = (1 + rets).cumprod()
dd = growth / growth.cummax() - 1

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
growth.plot(ax=axes[0], logy=True, title="Crecimiento de $1 (escala log)")
dd.plot(ax=axes[1], title="Drawdown")
plt.tight_layout()
plt.savefig("../reports/figures/01_growth_drawdown.png", dpi=150)
print(dd.min())   # peor caída de cada ETF

Ticker
SPY   -0.551895
QQQ   -0.534040
TLT   -0.483511
GLD   -0.455550
EEM   -0.664343
dtype: float64


### 2.2 Los retornos no son normales

In [3]:
spy = rets["SPY"]
print(f"curtosis en exceso: {stats.kurtosis(spy):.1f} | asimetría: {stats.skew(spy):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(spy, bins=100, stat="density", ax=ax[0])
x = np.linspace(spy.min(), spy.max(), 400)
ax[0].plot(x, stats.norm.pdf(x, spy.mean(), spy.std()), color="red", label="Normal")
ax[0].legend(); ax[0].set_title("Retornos diarios de SPY vs. normal")
stats.probplot(spy, dist="norm", plot=ax[1])
plt.tight_layout()
plt.savefig("../reports/figures/02_return_distribution.png", dpi=150)

curtosis en exceso: 15.4 | asimetría: -0.00


### 2.3 La volatilidad se agrupa

In [4]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
(spy.rolling(21).std() * np.sqrt(252)).plot(ax=ax[0], title="Volatilidad anualizada de SPY (21 d)")
plot_acf(spy.abs(), lags=40, ax=ax[1], title="Autocorrelación de |retorno|")
plt.tight_layout()
plt.savefig("../reports/figures/03_volatility_clustering.png", dpi=150)

### 2.4 Las correlaciones cambian con el régimen

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(rets.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax[0])
rets["SPY"].rolling(252).corr(rets["TLT"]).plot(ax=ax[1], title="Correlación móvil SPY–TLT (1 año)")
ax[1].axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.savefig("../reports/figures/04_correlations.png", dpi=150)

### Números de apoyo para las conclusiones

In [6]:
from statsmodels.tsa.stattools import acf
yrs = len(rets) / 252
summary = pd.DataFrame({
    "CAGR": growth.iloc[-1] ** (1 / yrs) - 1,
    "vol_anual": rets.std() * np.sqrt(252),
    "max_drawdown": dd.min(),
    "fecha_max_dd": dd.idxmin().dt.date,
    "curtosis_exceso": rets.apply(stats.kurtosis),
})
print(summary.round(3))
vol21 = spy.rolling(21).std() * np.sqrt(252)
print(f"\nVol. SPY 21d: mín {vol21.min():.1%}, mediana {vol21.median():.1%}, máx {vol21.max():.1%} ({vol21.idxmax().date()})")
print(f"ACF de SPY: retorno lag1 {acf(spy, nlags=1)[1]:.3f} | |retorno| lag1 {acf(spy.abs(), nlags=1)[1]:.3f}, lag20 {acf(spy.abs(), nlags=20)[20]:.3f}")
c = rets["SPY"].rolling(252).corr(rets["TLT"]).dropna()
print(f"Corr SPY–TLT: total {rets['SPY'].corr(rets['TLT']):.2f} | móvil mín {c.min():.2f} ({c.idxmin().date()}), máx {c.max():.2f} ({c.idxmax().date()})")
print("Corr SPY–TLT por periodo:", {p: round(rets.loc[a:b, 'SPY'].corr(rets.loc[a:b, 'TLT']), 2)
      for p, (a, b) in {"2005-2019": ("2005", "2019"), "2020": ("2020", "2020"), "2022-2025": ("2022", "2025")}.items()})

         CAGR  vol_anual  max_drawdown fecha_max_dd  curtosis_exceso
Ticker                                                              
SPY     0.107      0.191        -0.552   2009-03-09           15.360
QQQ     0.149      0.216        -0.534   2008-11-20            7.816
TLT     0.032      0.147        -0.484   2023-10-19            3.472
GLD     0.112      0.177        -0.456   2015-12-17            5.955
EEM     0.065      0.276        -0.664   2008-11-20           18.351

Vol. SPY 21d: mín 3.4%, mediana 12.9%, máx 94.5% (2008-10-28)
ACF de SPY: retorno lag1 -0.105 | |retorno| lag1 0.318, lag20 0.229
Corr SPY–TLT: total -0.31 | móvil mín -0.77 (2012-08-08), máx 0.32 (2024-07-01)
Corr SPY–TLT por periodo: {'2005-2019': np.float64(-0.41), '2020': np.float64(-0.48), '2022-2025': np.float64(0.09)}


## 2.5 Conclusiones del EDA (2005-2025)

**1. Colas gruesas: los retornos no son normales.** La curtosis en exceso de SPY es **15.4** (una normal tiene 0); la de EEM, 18.4. En el gráfico Q-Q, los puntos se alejan de la recta en los dos extremos: hubo días de −11 % y +14.5 % que una normal haría casi imposibles. **Consecuencia:** la desviación estándar y el Sharpe subestiman el riesgo de cola, así que reportaremos también el máximo drawdown y usaremos métodos que no supongan normalidad (bootstrap, PSR/DSR).

**2. La volatilidad se agrupa.** La volatilidad anualizada de SPY a 21 días va de 3.4 % a **94.5 %** (octubre de 2008), con mediana de 12.9 %. La autocorrelación de |retorno| es 0.32 al día siguiente y todavía 0.23 a 20 días, mientras que la del retorno con signo es apenas −0.10. El tamaño del movimiento es persistente; la dirección casi no. **Consecuencia:** tiene sentido probar la estrategia de volatilidad objetivo (Paso 3), y el bootstrap debe ser por bloques para respetar esta dependencia (Paso 7).

**3. Las correlaciones cambian con el régimen.** SPY–TLT tiene correlación −0.31 en todo el periodo, pero la móvil de un año va de **−0.77 (2012) a +0.32 (2024)**. Fue −0.41 en 2005-2019 y +0.09 en 2022-2025: los bonos dejaron de cubrir a las acciones justo en el entorno de inflación y tasas altas. **Consecuencia:** la diversificación entre acciones y bonos no es una constante y no se debe suponer en el análisis.

**4. El dolor de buy & hold.** Peor caída por ETF: SPY −55 % (marzo 2009), QQQ −53 %, EEM **−66 %**, GLD −46 % (2015) y TLT −48 % (octubre 2023, todavía alrededor de −40 % a finales de 2025). Un activo "defensivo" como TLT tuvo un drawdown comparable al de las acciones. CAGR 2005-2025: QQQ 14.9 %, GLD 11.2 %, SPY 10.7 %, EEM 6.5 %, TLT 3.2 %. Ese es el benchmark que una estrategia dice mejorar.

**Nota importante:** los retornos diarios son casi impredecibles (autocorrelación cercana a 0) y la señal que buscamos es pequeña frente al ruido. Por eso, en este proyecto la validación rigurosa importa más que la estrategia.
